# ema-second-moment — worked example 1: Single-Parameter v-Buffer: Three Manual EMA Steps

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-second-moment`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The Adam optimizer maintains a running average of squared gradients called the second moment, or `v`. Each step updates `v` using the rule `v = beta2 * v + (1 - beta2) * g**2`, where `beta2` is typically 0.999. This exponential moving average decays old squared-gradient history and accumulates new information. A large `beta2` means the buffer changes slowly — past history has strong influence.

## Worked solution

We start with `v = 0` (the standard Adam initialization) and a fixed gradient `g = 2.0`, then manually apply the update three times with `beta2 = 0.9`.

**Step 1:** `v_new = 0.9 * 0 + 0.1 * 4.0 = 0.4`. We multiply the old `v` by `beta2` (zero here since v is initialized to 0), and add `(1 - beta2) * g^2 = 0.1 * 4.0 = 0.4`.

**Step 2:** `v_new = 0.9 * 0.4 + 0.1 * 4.0 = 0.36 + 0.4 = 0.76`. The old v is 0.4; multiplying by 0.9 discounts it. We add the same 0.4 contribution from the current grad.

**Step 3:** `v_new = 0.9 * 0.76 + 0.1 * 4.0 = 0.684 + 0.4 = 1.084`. Each step, v grows toward `g^2 = 4.0` as the steady state. Notice the update is in-place: `v.copy_(...)` mutates the tensor object rather than rebinding the Python variable, which matters because an optimizer holds a reference to the same tensor throughout training.

In [ ]:
import torch as t

t.manual_seed(17)

# --- forward setup ---
beta2 = 0.9
g_val = 2.0

v = t.zeros(1)
g = t.tensor([g_val])

print(f"Initial v: {v.item():.6f}")
for step in range(1, 4):
    v.copy_(beta2 * v + (1 - beta2) * g.pow(2))
    print(f"After step {step}: v = {v.item():.6f}  (target g^2 = {g_val**2:.1f})")

# Verify convergence direction: v should be approaching g^2
assert v.item() < g_val**2, "v should still be below g^2 after 3 steps with beta2=0.9"
print("Done. v is converging toward g^2 =", g_val**2)